# Facial Emotion Recognition (FER2013)

Complete CNN pipeline: dataset loading, preprocessing, model definition, training and saving.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## 2. Load FER2013 CSV

In [ ]:
df = pd.read_csv("fer2013.csv")
pixels = df["pixels"].apply(lambda p: np.fromstring(p, dtype=int, sep=" "))
X = np.vstack(pixels.values).reshape(-1, 48, 48, 1).astype("float32") / 255.0
y = keras.utils.to_categorical(df["emotion"].values, num_classes=7)
print(X.shape, y.shape)

## 3. Build the CNN model

In [ ]:
inputs = keras.Input(shape=(48, 48, 1))

def conv_block(x, filters, dropout=0.25):
    x = layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(dropout)(x)
    return x

x = conv_block(inputs, 64)
x = conv_block(x, 128)
x = conv_block(x, 256)
x = conv_block(x, 512)
x = layers.Flatten()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(7, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train

In [ ]:
history = model.fit(
    X, y,
    epochs=40,
    batch_size=64,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5),
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    ],
)

## 5. Save the model

In [ ]:
model.save("emotion_model.h5")
print("Saved emotion_model.h5")